# Análisis e Imputación de Datos para Base de Datos Tomate - Quindío

#### Librerías

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from itertools import product 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate, GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import shap
from itertools import product

### Load Data

In [ ]:
BD_unificado_path = R"..\BaseDatos\BD Quindio unificados\Quindio_Movil_Manual_unificado.csv"
df_movil_manual_unificado = pd.read_csv(BD_unificado_path)

In [ ]:
df_movil_manual_unificado.head()

In [ ]:
df_movil_manual_unificado.describe()

### Eliminar Variables

In [ ]:
variables_a_eliminar = ['Diametro tallo (mm)','Posicion_x', 'Posicion_y', 
                        'x', 'y','Perímetro (mm)','Numero frutos','Fosforo_savia (ppm)',
                        'Fosforo_suelo_Hanna (ppm)','Frutos de 2da','RGB_capture',
                        'Tapo_capture','Parrot_capture','7in1_Conductivity[uS/cm]',
                        '7in1_Nitrogen[mg/kg]','7in1_Phosphorus[mg/kg]','7in1_Potasium[mg/kg]',
                        'Pynamometer_Radiation[W/m^2]']

df_reduced = df_movil_manual_unificado.drop(columns=variables_a_eliminar)
df_reduced['Fecha'] = pd.to_datetime(df_reduced['Fecha'], errors='coerce')
df_reduced = df_reduced.drop(columns=['Laboratorio','Cultivo','Planta'])

In [ ]:
df_reduced.info()

---
### Correlación # Frutos Cosechados vs Peso

In [ ]:
correlation_matrix = df_reduced.select_dtypes(include=[float,int]).corr()
plt.figure(figsize=(15, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.show()

In [ ]:
correlation_matrix = df_reduced[['Numero frutos cosechados ',
                                 'Peso frutos cosechados (Kg)']].select_dtypes(include=[float,int]).corr()
plt.figure(figsize=(15, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.show()

In [ ]:
# Crear nuevas columnas N, P y K extrayendo los valores después de cada letra
df_reduced['N'] = df_reduced['Tratamiento'].str.extract('N(\d+)', expand=False)
df_reduced['P'] = df_reduced['Tratamiento'].str.extract('P(\d+)', expand=False)
df_reduced['K'] = df_reduced['Tratamiento'].str.extract('K(\d+)', expand=False)

df_reduced['N'] = df_reduced['N'].astype(int)
df_reduced['P'] = df_reduced['P'].astype(int)
df_reduced['K'] = df_reduced['K'].astype(int)

### Filtrado de Valores Faltantes

In [ ]:
# Contar el número de valores NaN en cada fila
nan_counts = df_reduced.isna().sum(axis=1)
# Filtrar el DataFrame para mantener solo las filas con a lo sumo 21 valores NaN
df_filtered = df_reduced[nan_counts <= 24]

In [ ]:
df_filtered['Año'] = df_filtered['Fecha'].dt.year
df_filtered['Mes'] = df_filtered['Fecha'].dt.month
df_filtered['Día'] = df_filtered['Fecha'].dt.day

### _Imputación multivariante por tratamiento_

##### 6.4.3. Imputación de características multivariadas 

Un enfoque más sofisticado es utilizar la IterativeImputerclase, que modela cada característica con valores faltantes como una función de otras características, y utiliza esa estimación para la imputación. Lo hace de manera iterativa: en cada paso, una columna de característica se designa como salida $y$ y las otras columnas de características se tratan como entradas $X$. Se ajusta un regresor para los valores conocidos . Luego, el regresor se utiliza para predecir los valores faltantes de . Esto se hace para cada característica de manera iterativa y luego se repite para las rondas de imputación. Se devuelven los resultados de la ronda de imputación final.$(X, y)$ y $y$ $max_{iter}$

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
from sklearn.ensemble import RandomForestRegressor

# Paso 1: Extraer número de tratamiento
df_filtered['Tratamiento_num'] = df_filtered['Tratamiento'].str.extract(r'T(\d+)').astype(float)

# Paso 2: Seleccionar columnas numéricas con al menos 5% de datos válidos
min_frac_no_nulos = 0.05
df_numeric = df_filtered.select_dtypes(include=[float, int])
df_numeric = df_numeric.loc[:, df_numeric.isna().mean() < (1 - min_frac_no_nulos)]

# Paso 3: Guardar percentiles para clipping (1% y 99%) antes de imputación
limites_clip = df_numeric.quantile([0.01, 0.99]).T
limites_clip.columns = ['min_val', 'max_val']

# GUARDAR EL DATAFRAME ORIGINAL COMPLETO ANTES DE CUALQUIER IMPUTACIÓN
df_numeric_original = df_numeric.copy()

def imputar_grupo(group):
    columnas_validas = group.columns[~group.isna().all()]
    group_valido = group[columnas_validas]
    
    # GUARDAR MÁSCARA DE VALORES ORIGINALES
    mascara_original = ~group_valido.isna()
    valores_originales = group_valido.copy()
    
    imputer = IterativeImputer(estimator=RandomForestRegressor(n_estimators=10, random_state=42), 
                               max_iter=20, random_state=42)
    imputado = imputer.fit_transform(group_valido)
    
    df_imputado = pd.DataFrame(imputado, columns=columnas_validas, index=group.index)
    
    # RESTAURAR VALORES ORIGINALES (solo imputar los NaN)
    df_imputado[mascara_original] = valores_originales[mascara_original]
    
    df_resultado = pd.DataFrame(index=group.index, columns=group.columns)
    df_resultado[columnas_validas] = df_imputado
    return df_resultado

# Paso 5: Imputación por grupo sin columna de agrupamiento
tratamiento_col = df_filtered['Tratamiento_num']
df_numeric['Tratamiento_num'] = tratamiento_col

# CREAR df_imputed SIN PERDER ÍNDICES
df_imputed = pd.DataFrame(index=df_numeric.index, columns=df_numeric.columns)

# Imputar por grupo MANTENIENDO LOS ÍNDICES ORIGINALES
for tratamiento in df_numeric['Tratamiento_num'].unique():
    mask = df_numeric['Tratamiento_num'] == tratamiento
    grupo = df_numeric.loc[mask].drop(columns=['Tratamiento_num'])
    grupo_imputado = imputar_grupo(grupo)
    df_imputed.loc[mask] = grupo_imputado

# Restaurar columna de tratamiento
df_imputed['Tratamiento_num'] = tratamiento_col

# Paso 6: Reimputación si hay negativos
max_iter_reimputacion = 11
i = 0
while (df_imputed < 0).any().any() and i < max_iter_reimputacion:
    print(f"🔄 Reimputación {i+1}: Se encontraron valores negativos. Reimputando...")
    
    # Reemplazar negativos con los valores ORIGINALES si existían, o NaN si eran imputados
    for col in df_imputed.columns:
        if col != 'Tratamiento_num':
            mask_negativo = df_imputed[col] < 0
            # Si el valor original existía, restaurarlo; si no, poner NaN
            df_imputed.loc[mask_negativo, col] = df_numeric_original.loc[mask_negativo, col].fillna(np.nan)
    
    # Reimputar solo los NaN (incluyendo los negativos que se convirtieron en NaN)
    for tratamiento in df_imputed['Tratamiento_num'].unique():
        mask = df_imputed['Tratamiento_num'] == tratamiento
        grupo = df_imputed.loc[mask].drop(columns=['Tratamiento_num'])
        
        # Solo reimputar si hay NaN
        if grupo.isna().any().any():
            grupo_imputado = imputar_grupo(grupo)
            df_imputed.loc[mask, grupo_imputado.columns] = grupo_imputado
    
    i += 1

if (df_imputed < 0).any().any():
    print("⚠ Aún quedan valores negativos después de varias reimputaciones.")
else:
    print("✅ No quedan valores negativos en el DataFrame imputado.")

# Paso 7: Clipping (SOLO EN VALORES IMPUTADOS, NO EN ORIGINALES)
for col in df_imputed.columns:
    if col in limites_clip.index and col != 'Tratamiento_num':
        min_val = limites_clip.loc[col, 'min_val']
        max_val = limites_clip.loc[col, 'max_val']
        
        # Solo hacer clipping en valores que ERAN NaN originalmente
        mascara_imputados = df_numeric_original[col].isna()
        df_imputed.loc[mascara_imputados, col] = df_imputed.loc[mascara_imputados, col].clip(lower=min_val, upper=max_val)

print("🧪 Listo: se aplicó recorte de valores extremos (clip) usando percentiles 1% y 99%.")

# VERIFICACIÓN FINAL
print(f"\n✅ Forma del dataframe original: {df_numeric_original.shape}")
print(f"✅ Forma del dataframe imputado: {df_imputed.shape}")
print(f"✅ Índices son iguales: {df_numeric_original.index.equals(df_imputed.index)}")

In [ ]:
mapeo = {
    "Altura planta (cm)": "Plant_Height (cm)",
    "Clorofila (SPAD)": "Chlorophyll (SPAD)",
    "Numero flores ": "Number of Flowers",
    "Numero frutos cosechados ": "Number of Harvested Fruits",
    "Peso frutos cosechados (Kg)": "Weight of Harvested Fruits (Kg)",
    "Tamanno altura (mm)": "Fruit Height (mm)",
    "pH_savia": "Sap pH",
    "K_savia (ppm)": "Sap K (ppm)",
    "Ca_savia (ppm)": "Sap Ca (ppm)",
    "Na_savia (ppm)": "Sap Na (ppm)",
    "NO3_savia (ppm)": "Sap NO3 (ppm)",
    "Conductividad_savia (mS/cm)": "Sap Conductivity (mS/cm)",
    "pH_suelo_Horiba": "Soil pH Horiba",
    "K_suelo_Horiba (ppm)": "Soil K Horiba (ppm)",
    "Ca_suelo_Horiba (ppm)": "Soil Ca Horiba (ppm)",
    "Na_suelo_Horiba (ppm)": "Soil Na Horiba (ppm)",
    "NO3_suelo_Horiba (ppm)": "Soil NO3 Horiba (ppm)",
    "Conductividad_suelo_Horiba (mS/cm)": "Soil Conductivity Horiba (mS/cm)",
    "Tamanno cintura (mm)": "Fruit Diameter (mm)",
    "N": "Nitrogen",
    "P": "Phosphorus",
    "K": "Potassium",
    "Año": "Year",
    "Mes": "Month",
    "Día": "Day",
    "Tratamiento_num": "Treatment_Num"
}

df_imputed.rename(columns=mapeo, inplace=True)
df_filtered.rename(columns=mapeo, inplace=True)
df_numeric.rename(columns=mapeo, inplace=True)

In [ ]:
df_imputed.info() 

In [ ]:
# guardar df_imputed como csv
df_imputed.to_csv("../BaseDatos/df_imputed.csv", index=False)